In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd

In [7]:
path1 = '/content/drive/MyDrive/Python for ML/Build Model/cleaned_all_phones_final.csv'
path2 = '/content/drive/MyDrive/Python for ML/Build Model/price_final.csv'
df = pd.read_csv(path1)
df_price = pd.read_csv(path2)

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price(USD)'])
y = df_price['price(USD)']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)

    # Metrics trên dữ liệu chuẩn hóa
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    print(f"--- {name} ---")
    print(f"Best MAE:  {mae:.4f}")
    print(f"Best MSE:  {mse:.4f}")
    print(f"Best RMSE: {rmse:.4f}")


In [10]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
evaluate_model(lr_model, X_test, y_test, "Linear Regression")

--- Linear Regression ---
Best MAE:  149.5826
Best MSE:  54558.4878
Best RMSE: 233.5776


In [11]:
from sklearn.linear_model import Ridge

param_grid_ridge = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100, 200]
}

best_mae_ridge = float('inf')
best_params_ridge = None

for alpha in param_grid_ridge['alpha']:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    if mae < best_mae_ridge:
        best_mae_ridge = mae
        best_params_ridge = {'alpha': alpha}
        best_ridge_model = model

print("Best Ridge Params:", best_params_ridge)
evaluate_model(best_ridge_model, X_test, y_test, "Ridge Regression")

Best Ridge Params: {'alpha': 0.001}
--- Ridge Regression ---
Best MAE:  149.5828
Best MSE:  54558.5716
Best RMSE: 233.5778


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
# Dự đoán và đánh giá
evaluate_model(rf_model, X_test, y_test, "Random Forest (No Tuning)")

--- Random Forest (No Tuning) ---
Best MAE:  145.3581
Best MSE:  65940.4533
Best RMSE: 256.7887


In [13]:
param_grid_rf = {
    'n_estimators': [50, 75, 100, 125, 150, 175],
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 3, 4, 5, 8, 10],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}

best_mae_rf = float('inf')
best_params_rf = None

for n in param_grid_rf['n_estimators']:
    for depth in param_grid_rf['max_depth']:
        for split in param_grid_rf['min_samples_split']:
            for leaf in param_grid_rf['min_samples_leaf']:
                model = RandomForestRegressor(n_estimators=n, max_depth=depth,
                                               min_samples_split=split, min_samples_leaf=leaf,
                                               random_state=42, n_jobs=-1)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                mae = mean_absolute_error(y_test, y_pred)
                if mae < best_mae_rf:
                    best_mae_rf = mae
                    best_params_rf = {
                        'n_estimators': n, 'max_depth': depth,
                        'min_samples_split': split, 'min_samples_leaf': leaf
                    }
                    best_rf_model = model

print("Best RF Params:", best_params_rf)
evaluate_model(best_rf_model, X_test, y_test, "Random Forest")


Best RF Params: {'n_estimators': 175, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 5}
--- Random Forest ---
Best MAE:  132.1540
Best MSE:  50496.6097
Best RMSE: 224.7145


In [14]:
from xgboost import XGBRegressor

# Mô hình XGBoost với tham số mặc định
xgb_model = XGBRegressor(n_jobs=-1, random_state=42)
xgb_model.fit(X_train, y_train)

# Đánh giá mô hình
evaluate_model(xgb_model, X_test, y_test, "XGBoost (No Tuning)")

--- XGBoost (No Tuning) ---
Best MAE:  153.8969
Best MSE:  79559.6121
Best RMSE: 282.0631


In [15]:
from xgboost import XGBRegressor

param_grid_xgb = {
    'n_estimators': [100, 125, 150, 175, 200],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'max_depth': [4, 5, 6, 7, 8],
    'subsample': [0.5, 0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7,0.9, 1.0],
    'gamma': [0.05, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1],
    'reg_lambda': [0.1, 0.3, 0.5, 0.7, 1]
}

best_mae_xgb = float('inf')
best_params_xgb = None

for n in param_grid_xgb['n_estimators']:
    for lr in param_grid_xgb['learning_rate']:
        for depth in param_grid_xgb['max_depth']:
            for sub in param_grid_xgb['subsample']:
                model = XGBRegressor(n_estimators=n, learning_rate=lr, max_depth=depth,
                                     subsample=sub, n_jobs=-1, random_state=42)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                mae = mean_absolute_error(y_test, y_pred)
                if mae < best_mae_xgb:
                    best_mae_xgb = mae
                    best_params_xgb = {
                        'n_estimators': n, 'learning_rate': lr,
                        'max_depth': depth, 'subsample': sub
                    }
                    best_xgb_model = model

print("Best XGBoost Params:", best_params_xgb)
evaluate_model(best_xgb_model, X_test, y_test, "XGBoost")


Best XGBoost Params: {'n_estimators': 125, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.5}
--- XGBoost ---
Best MAE:  131.3202
Best MSE:  50811.5827
Best RMSE: 225.4142


In [16]:
from sklearn.svm import SVR

svr_model = SVR()
svr_model.fit(X_train, y_train)

# Đánh giá mô hình
evaluate_model(svr_model, X_test, y_test, "SVR (No Tuning)")

--- SVR (No Tuning) ---
Best MAE:  162.1648
Best MSE:  78293.7393
Best RMSE: 279.8102


In [17]:
from sklearn.svm import SVR

param_grid_svr = {
    'C': [0.5, 0.7, 1, 2, 3, 5, 7, 10],
    'epsilon': [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5],
    'gamma': ['scale', 'auto', 0.001, 0.003, 0.01, 0.03, 0.05, 0.1],
    'kernel': ['rbf']
}


best_mae_svr = float('inf')
best_params_svr = None

for C in param_grid_svr['C']:
    for eps in param_grid_svr['epsilon']:
        for gamma in param_grid_svr['gamma']:
            model = SVR(C=C, epsilon=eps, kernel='rbf', gamma=gamma)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            mae = mean_absolute_error(y_test, y_pred)
            if mae < best_mae_svr:
                best_mae_svr = mae
                best_params_svr = {'C': C, 'epsilon': eps, 'gamma': gamma}
                best_svr_model = model

print("Best SVR Params:", best_params_svr)
evaluate_model(best_svr_model, X_test, y_test, "SVR")


Best SVR Params: {'C': 10, 'epsilon': 0.5, 'gamma': 0.03}
--- SVR ---
Best MAE:  145.6744
Best MSE:  67266.4690
Best RMSE: 259.3578


In [18]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)

# Đánh giá mô hình
evaluate_model(gb_model, X_test, y_test, "Gradient Boosting (No Tuning)")


--- Gradient Boosting (No Tuning) ---
Best MAE:  134.4026
Best MSE:  49073.1785
Best RMSE: 221.5247


In [19]:
from sklearn.ensemble import GradientBoostingRegressor

param_grid_gb = {
    'n_estimators': [100, 125, 150, 175, 200],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5, 7],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4],
    'subsample': [0.8, 0.9, 1.0]
}

best_mae_gb = float('inf')
best_params_gb = None

for n in param_grid_gb['n_estimators']:
    for lr in param_grid_gb['learning_rate']:
        for depth in param_grid_gb['max_depth']:
            for split in param_grid_gb['min_samples_split']:
                for leaf in param_grid_gb['min_samples_leaf']:
                    for sub in param_grid_gb['subsample']:
                        model = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                                          max_depth=depth, min_samples_split=split,
                                                          min_samples_leaf=leaf, subsample=sub,
                                                          random_state=42)
                        model.fit(X_train, y_train)
                        y_pred = model.predict(X_test)
                        mae = mean_absolute_error(y_test, y_pred)
                        if mae < best_mae_gb:
                            best_mae_gb = mae
                            best_params_gb = {
                                'n_estimators': n, 'learning_rate': lr,
                                'max_depth': depth, 'min_samples_split': split,
                                'min_samples_leaf': leaf, 'subsample': sub
                            }
                            best_gb_model = model

print("Best Gradient Boosting Params:", best_params_gb)
evaluate_model(best_gb_model, X_test, y_test, "Gradient Boosting")

Best Gradient Boosting Params: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'min_samples_split': 5, 'min_samples_leaf': 2, 'subsample': 0.8}
--- Gradient Boosting ---
Best MAE:  133.0229
Best MSE:  48560.3384
Best RMSE: 220.3641


# DATASET KHÔNG SCALER

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
path = '/content/drive/MyDrive/Python for ML/Build Model/cleaned_all_phones_normal.csv'
df = pd.read_csv(path)
df.head()

,inches,ram(GB),weight(g),storage(GB),video_720p,video_4K,video_8K,video_60fps,video_120fps,video_240fps,video_960fps,width,height,total_cam_func,brand_group,os_group_encoded,brand_Apple,log_ppi,log_price
0,5.0,2,140.0,16,1,0,0,0,0,0,0,720,1280,2,403.913650,0,0,5.682630,4.787492
1,5.3,2,140.0,16,0,0,0,0,0,0,0,720,1280,2,264.000000,0,0,5.624361,4.605170
2,5.2,4,147.0,16,0,0,0,0,0,0,0,1080,1920,2,403.913650,0,0,6.048875,6.040255
3,5.5,4,165.0,32,1,0,0,0,1,0,0,1080,1920,4,306.254129,0,0,5.992785,5.010635
4,5.1,4,145.0,32,1,1,0,1,0,0,0,1080,1920,5,403.913650,0,0,6.068293,6.040255


In [23]:
df = df.applymap(lambda x: int(x) if type(x) == bool else x)
df.head()

<ipython-input-23-8f435bab8115>:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: int(x) if type(x) == bool else x)


,inches,ram(GB),weight(g),storage(GB),video_720p,video_4K,video_8K,video_60fps,video_120fps,video_240fps,video_960fps,width,height,total_cam_func,brand_group,os_group_encoded,brand_Apple,log_ppi,log_price
0,5.0,2,140.0,16,1,0,0,0,0,0,0,720,1280,2,403.913650,0,0,5.682630,4.787492
1,5.3,2,140.0,16,0,0,0,0,0,0,0,720,1280,2,264.000000,0,0,5.624361,4.605170
2,5.2,4,147.0,16,0,0,0,0,0,0,0,1080,1920,2,403.913650,0,0,6.048875,6.040255
3,5.5,4,165.0,32,1,0,0,0,1,0,0,1080,1920,4,306.254129,0,0,5.992785,5.010635
4,5.1,4,145.0,32,1,1,0,1,0,0,0,1080,1920,5,403.913650,0,0,6.068293,6.040255


In [38]:
def evaluate_model(model, X_test, y_test, name):
    y_pred_log = model.predict(X_test)

    # Chuyển từ log(price) về price
    y_pred = np.exp(y_pred_log)
    y_true = np.exp(y_test)

    # Tính các metric trên giá gốc (USD)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    print(f"--- {name} ---")
    print(f"MAE (USD):  {mae:.2f}")
    print(f"MSE (USD):  {mse:.2f}")
    print(f"RMSE (USD): {rmse:.2f}")


In [39]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['log_price'])
y = df['log_price']

# Chia train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [40]:
from xgboost import XGBRegressor

# Mô hình XGBoost với tham số mặc định
xgb_model = XGBRegressor(n_jobs=-1, random_state=42)
xgb_model.fit(X_train, y_train)

# Đánh giá mô hình
evaluate_model(xgb_model, X_test, y_test, "XGBoost (No Tuning)")

--- XGBoost (No Tuning) ---
MAE (USD):  149.66
MSE (USD):  74909.87
RMSE (USD): 273.70


In [41]:
from xgboost import XGBRegressor

param_grid_xgb = {
    'n_estimators': [100, 125, 150, 175, 200],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'max_depth': [4, 5, 6, 7, 8],
    'subsample': [0.5, 0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7,0.9, 1.0],
    'gamma': [0.05, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1],
    'reg_lambda': [0.1, 0.3, 0.5, 0.7, 1]
}

best_mae_xgb = float('inf')
best_params_xgb = None

for n in param_grid_xgb['n_estimators']:
    for lr in param_grid_xgb['learning_rate']:
        for depth in param_grid_xgb['max_depth']:
            for sub in param_grid_xgb['subsample']:
                model = XGBRegressor(n_estimators=n, learning_rate=lr, max_depth=depth,
                                     subsample=sub, n_jobs=-1, random_state=42)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                mae = mean_absolute_error(y_test, y_pred)
                if mae < best_mae_xgb:
                    best_mae_xgb = mae
                    best_params_xgb = {
                        'n_estimators': n, 'learning_rate': lr,
                        'max_depth': depth, 'subsample': sub
                    }
                    best_xgb_model = model

print("Best XGBoost Params:", best_params_xgb)
evaluate_model(best_xgb_model, X_test, y_test, "XGBoost")

Best XGBoost Params: {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 6, 'subsample': 0.5}
--- XGBoost ---
MAE (USD):  119.62
MSE (USD):  48801.63
RMSE (USD): 220.91


In [42]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
# Dự đoán và đánh giá
evaluate_model(rf_model, X_test, y_test, "Random Forest (No Tuning)")

--- Random Forest (No Tuning) ---
MAE (USD):  131.71
MSE (USD):  56648.73
RMSE (USD): 238.01


In [43]:
param_grid_rf = {
    'n_estimators': [50, 75, 100, 125, 150, 175],
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 3, 4, 5, 8, 10],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}

best_mae_rf = float('inf')
best_params_rf = None

for n in param_grid_rf['n_estimators']:
    for depth in param_grid_rf['max_depth']:
        for split in param_grid_rf['min_samples_split']:
            for leaf in param_grid_rf['min_samples_leaf']:
                model = RandomForestRegressor(n_estimators=n, max_depth=depth,
                                               min_samples_split=split, min_samples_leaf=leaf,
                                               random_state=42, n_jobs=-1)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                mae = mean_absolute_error(y_test, y_pred)
                if mae < best_mae_rf:
                    best_mae_rf = mae
                    best_params_rf = {
                        'n_estimators': n, 'max_depth': depth,
                        'min_samples_split': split, 'min_samples_leaf': leaf
                    }
                    best_rf_model = model

print("Best RF Params:", best_params_rf)
evaluate_model(best_rf_model, X_test, y_test, "Random Forest")

Best RF Params: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 5}
--- Random Forest ---
MAE (USD):  123.87
MSE (USD):  50299.81
RMSE (USD): 224.28


In [44]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(random_state=42)
gb_model.fit(X_train, y_train)

# Đánh giá mô hình
evaluate_model(gb_model, X_test, y_test, "Gradient Boosting (No Tuning)")

--- Gradient Boosting (No Tuning) ---
MAE (USD):  124.14
MSE (USD):  48304.74
RMSE (USD): 219.78


In [45]:
from sklearn.ensemble import GradientBoostingRegressor

param_grid_gb = {
    'n_estimators': [100, 125, 150, 175, 200],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5, 7],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4],
    'subsample': [0.8, 0.9, 1.0]
}

best_mae_gb = float('inf')
best_params_gb = None

for n in param_grid_gb['n_estimators']:
    for lr in param_grid_gb['learning_rate']:
        for depth in param_grid_gb['max_depth']:
            for split in param_grid_gb['min_samples_split']:
                for leaf in param_grid_gb['min_samples_leaf']:
                    for sub in param_grid_gb['subsample']:
                        model = GradientBoostingRegressor(n_estimators=n, learning_rate=lr,
                                                          max_depth=depth, min_samples_split=split,
                                                          min_samples_leaf=leaf, subsample=sub,
                                                          random_state=42)
                        model.fit(X_train, y_train)
                        y_pred = model.predict(X_test)
                        mae = mean_absolute_error(y_test, y_pred)
                        if mae < best_mae_gb:
                            best_mae_gb = mae
                            best_params_gb = {
                                'n_estimators': n, 'learning_rate': lr,
                                'max_depth': depth, 'min_samples_split': split,
                                'min_samples_leaf': leaf, 'subsample': sub
                            }
                            best_gb_model = model

print("Best Gradient Boosting Params:", best_params_gb)
evaluate_model(best_gb_model, X_test, y_test, "Gradient Boosting")

Best Gradient Boosting Params: {'n_estimators': 125, 'learning_rate': 0.05, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4, 'subsample': 0.9}
--- Gradient Boosting ---
MAE (USD):  119.24
MSE (USD):  47666.18
RMSE (USD): 218.33


In [46]:
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np

# Base models
base_models = [
    ('xgb', XGBRegressor(n_estimators=100, random_state=42)),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('lr', LinearRegression()),
    ('ridge', Ridge(alpha=1.0)),
    ('lasso', Lasso(alpha=0.1)),
    ('knn', KNeighborsRegressor(n_neighbors=5)),
    ('svr', SVR(C=1.0, epsilon=0.2)),
    ('extra', ExtraTreesRegressor(n_estimators=100, random_state=42))
]

# Meta-model
meta_model = GradientBoostingRegressor(random_state=42)

# Stacking model
stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    passthrough=True
)

# Train
stack_model.fit(X_train, y_train)

# Evaluate
evaluate_model(stack_model, X_test, y_test, "Stacking")

--- Stacking ---
MAE (USD):  126.72
MSE (USD):  50080.50
RMSE (USD): 223.79
